In [10]:
import pandas as pd 
import numpy as np

In [11]:
np.random.seed(42)

In [12]:
#create empty containers
cycles = []
time = []
voltage = []
current = []
temperature = []

In [13]:
# generate Data aise hi khud se like battery 
for cycle in range (1,101):
    for t in range (0,50):
        cycles.append(cycle)
        time.append(t)

        base_voltage = 4.2 - 0.01* cycle
        voltage.append(base_voltage - 0.02 * t + np.random.normal(0,0.01))

        current.append(-1.5)
        temperature.append(25+0.02* cycle + np.random.normal(0,0.2))

In [14]:
# convert to DataFrame
df_large = pd.DataFrame({
    "cycle": cycles,
    "time": time,
    "voltage": voltage,
    "current": current,
    "temperature": temperature
})

In [15]:
df_large.shape

(5000, 5)

In [16]:
# lets simulate capacity per cycle
capacity_per_cycle = 2.5 - 0.005 * df_large["cycle"].unique()

In [17]:
df_target = pd.DataFrame({
    "cycle": df_large["cycle"].unique(),
    "capacity": capacity_per_cycle
})

In [18]:
df_target.head()

,cycle,capacity
0,1,2.495
1,2,2.490
2,3,2.485
3,4,2.480
4,5,2.475


In [19]:
df_early = df_large[df_large["cycle"]<=10]

In [20]:
df_early.shape

(500, 5)

In [21]:
df_early.head()


,cycle,time,voltage,current,temperature
0,1,0,4.194967,-1.5,24.992347
1,1,1,4.176477,-1.5,25.324606
2,1,2,4.147658,-1.5,24.973173
3,1,3,4.145792,-1.5,25.173487
4,1,4,4.105305,-1.5,25.128512


In [22]:
grouped_early = df_large.groupby("cycle")

In [23]:
grouped_early.head()

,cycle,time,voltage,current,temperature
0,1,0,4.194967,-1.5,24.992347
1,1,1,4.176477,-1.5,25.324606
2,1,2,4.147658,-1.5,24.973173
3,1,3,4.145792,-1.5,25.173487
4,1,4,4.105305,-1.5,25.128512
...,...,...,...,...,...
4950,100,0,3.209347,-1.5,27.073985
4951,100,1,3.168827,-1.5,26.991702
4952,100,2,3.169705,-1.5,26.748347
4953,100,3,3.146366,-1.5,27.464522


In [24]:
min_voltage = grouped_early["voltage"].min()

In [25]:
avg_temp = grouped_early["temperature"].mean()

In [26]:
voltage_slope = grouped_early["voltage"].apply(
    lambda x: (x.iloc[-1] - x.iloc[0])/ len(x)
)

In [27]:
df_early_features = pd.DataFrame({
    "cycle": min_voltage.index,
    "min_voltage": min_voltage.values,
    "avg_temperature": avg_temp.values,
    "voltage_slope": voltage_slope.values
    })

In [28]:
df_early_features.head()

,cycle,min_voltage,avg_temperature,voltage_slope
0,1,3.210051,25.005597,-0.019698
1,2,3.200582,25.068012,-0.019305
2,3,3.198129,25.060169,-0.019509
3,4,3.178855,25.097226,-0.019457
4,5,3.161244,25.080075,-0.019456


In [29]:
early_summary = pd.DataFrame({
    "mean_min_voltage": [df_early_features["min_voltage"].mean()],
    "mean_voltage_slop": [df_early_features["voltage_slope"].mean()],
    "mean_temperature": [df_early_features["avg_temperature"].mean()]
})

In [30]:
early_summary

,mean_min_voltage,mean_voltage_slop,mean_temperature
0,2.714506,-0.019597,26.010103


In [31]:
long_term_capacity = df_target[df_target["cycle"] == 100]["capacity"].values[0]

In [32]:
long_term_capacity

np.float64(2.0)

In [33]:
early_summary["target_capacity_cycle100"] = long_term_capacity

In [34]:
early_summary

,mean_min_voltage,mean_voltage_slop,mean_temperature,target_capacity_cycle100
0,2.714506,-0.019597,26.010103,2.0


In [35]:
early_summary.to_csv(
    "../../data/week3_day2_early_features.csv",
    index=False
)